##  TechMind — Preparación del **Dataset final**

#### Equipo tejONEs

#### 05_exploracion_dataset_final.ipynb

- El proceso general que sigue este pipeline es:
    - [x]  Carga de los datasets procesados de Coursera, Microsoft Learn, OpenAlex y StackExchange
    - [x]  Validación del esquema común: `titulo`, `texto`, `categoria`, `autor`, `tipo`
    - [x]  Unificación de fuentes y filtro a las siete categorías oficiales del proyecto
    - [x]  Eliminación de duplicados entre fuentes mediante título y texto normalizados
    - [x]  Verificación de disponibilidad y selección determinista, sin reemplazo, de 200 registros por categoría
    - [x]  Auditoría de calidad y validación del total esperado de 1,400 registros
    - [x]  Exportación del dataset unificado final en la carpeta `procesados/`

## Importaciones

In [2]:
import os
import re
from pathlib import Path

# Dependencias en requirements.txt
import nltk
import pandas as pd
from deep_translator import GoogleTranslator
from nltk.corpus import stopwords
from tqdm.notebook import tqdm

# Descargar recursos de NLP
nltk.download('stopwords', quiet=True)
spanish_stopwords = set(stopwords.words('spanish'))

## Configuración

In [3]:
def find_project_root(start: Path) -> Path:
    """Encuentra la raíz del proyecto desde el directorio actual o sus padres."""
    for candidate in [start, *start.parents]:
        if (candidate / 'data_science').exists() and (candidate / 'README.md').exists():
            return candidate
    return start


base_dir = Path.cwd().resolve()
project_root = find_project_root(base_dir)

CARPETA_DATA = str((project_root / 'data_science' / 'data').resolve())
CARPETA_CRUDOS = str((project_root / 'data_science' / 'data' / 'crudos').resolve())
CARPETA_PROCESADOS = str((project_root / 'data_science' / 'data' / 'procesados').resolve())


print(f'✅ Ruta de procesados existe: {Path(CARPETA_PROCESADOS).exists()}')
print(f'✅ Ruta de crudos existe: {Path(CARPETA_CRUDOS).exists()}')
print(f'✅ Ruta de datos existe: {Path(CARPETA_DATA).exists()}')
print(f'📂 Datos procesados: {CARPETA_PROCESADOS}')
print(f'📂 Datos crudos: {CARPETA_CRUDOS}')
print(f'📁 Proyecto local: {CARPETA_DATA}')

✅ Ruta de procesados existe: True
✅ Ruta de crudos existe: True
✅ Ruta de datos existe: True
📂 Datos procesados: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\procesados
📂 Datos crudos: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\crudos
📁 Proyecto local: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data


# 3.Unificar los cuatro datasets para obtener 200 registros por categoría

##  3.1.Carga de los cuatro datasets

In [4]:
DATASET_PATHS = {
    'coursera': f'{CARPETA_PROCESADOS}/dataset_FINAL_coursera.csv',
    'mslearn': f'{CARPETA_PROCESADOS}/dataset_FINAL_mslearn.csv',
    'openalex': f'{CARPETA_PROCESADOS}/dataset_FINAL_openalex.csv',
    'stackexchange': f'{CARPETA_PROCESADOS}/dataset_FINAL_stackexchange.csv',
}
FINAL_COLUMNS = ['titulo', 'texto', 'categoria', 'autor', 'tipo']

def read_csv_with_fallback(file_path):
    # Carga segura con alternativa de codificación (encoding fallback) en caso de error
    try:
        return pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(file_path, encoding='latin1')

datasets = {}
for source, file_path in DATASET_PATHS.items():
    if not Path(file_path).exists():
        raise FileNotFoundError(f'Falta ejecutar el notebook fuente; no existe: {file_path}')

    df_source = read_csv_with_fallback(file_path)
    missing_columns = [column for column in FINAL_COLUMNS if column not in df_source.columns]
    if missing_columns:
        raise ValueError(f'{source} no contiene las columnas requeridas: {missing_columns}')

    df_source = df_source[FINAL_COLUMNS].copy()
    df_source['fuente'] = source
    datasets[source] = df_source
    print(f'✅ {source}: {len(df_source)} registros cargados.')

✅ coursera: 343 registros cargados.
✅ mslearn: 244 registros cargados.
✅ openalex: 463 registros cargados.
✅ stackexchange: 350 registros cargados.


## 3.2 Unificación y selección de 200 registros por categoría

In [5]:
REQUIRED_CATEGORIES = [
    'Backend', 'Bases de Datos', 'Cloud', 'Data Science',
    'Frontend', 'Mobile', 'DevOps',
]
TARGET_FINAL_PER_CATEGORY = 200

# 1. Unificar los cuatro datasets
df_unified = pd.concat(datasets.values(), ignore_index=True)
df_unified = df_unified[df_unified['categoria'].isin(REQUIRED_CATEGORIES)].reset_index(drop=True)

# 2. Eliminar duplicados exactos mediante claves normalizadas de título y texto
df_unified['_titulo_key'] = (
    df_unified['titulo'].fillna('').astype(str).str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
)
df_unified['_texto_key'] = (
    df_unified['texto'].fillna('').astype(str).str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
)

before_dedup = len(df_unified)
df_unified = df_unified.drop_duplicates(subset=['_titulo_key', '_texto_key']).reset_index(drop=True)
print(f'Duplicados transversales eliminados: {before_dedup - len(df_unified)}')

print('\n=== APORTE DISPONIBLE POR FUENTE Y CATEGORÍA ===')
display(pd.crosstab(df_unified['categoria'], df_unified['fuente']).reindex(REQUIRED_CATEGORIES, fill_value=0))

# 3. Verificar disponibilidad antes de seleccionar exactamente 200 registros
available_counts = df_unified['categoria'].value_counts().reindex(REQUIRED_CATEGORIES, fill_value=0)
missing_counts = {
    category: TARGET_FINAL_PER_CATEGORY - int(available_counts[category])
    for category in REQUIRED_CATEGORIES
    if available_counts[category] < TARGET_FINAL_PER_CATEGORY
}

if missing_counts:
    raise ValueError(f'No hay suficientes registros únicos para completar 200 por categoría: {missing_counts}')

# 4. Selección determinista y sin reemplazo
df_final_unified = pd.concat([
    df_unified[df_unified['categoria'] == category].sample(
        n=TARGET_FINAL_PER_CATEGORY, random_state=42, replace=False
    )
    for category in REQUIRED_CATEGORIES
]).reset_index(drop=True)

Duplicados transversales eliminados: 0

=== APORTE DISPONIBLE POR FUENTE Y CATEGORÍA ===


fuente,coursera,mslearn,openalex,stackexchange
categoria,,,,
Backend,50,45,55,50
Bases de Datos,50,50,50,50
Cloud,50,50,50,50
Data Science,50,50,50,50
Frontend,50,21,79,50
Mobile,50,4,96,50
DevOps,43,24,83,50


##  3.3.Exportación final y auditoría de calidad

In [6]:
df_final_unified = df_final_unified[FINAL_COLUMNS]

print("=== DISTRIBUCIÓN FINAL DEL DATASET UNIFICADO ===")
category_counts = df_final_unified['categoria'].value_counts()
print(category_counts)

# Validación estricta: exactamente 200 registros en cada una de las siete categorías
expected_counts = pd.Series(TARGET_FINAL_PER_CATEGORY, index=REQUIRED_CATEGORIES, name='registros')
actual_counts = category_counts.reindex(REQUIRED_CATEGORIES, fill_value=0)
assert actual_counts.to_dict() == expected_counts.to_dict(), (actual_counts.to_dict(), expected_counts.to_dict())
assert len(df_final_unified) == TARGET_FINAL_PER_CATEGORY * len(REQUIRED_CATEGORIES)
assert df_final_unified.columns.tolist() == FINAL_COLUMNS
print(f"\n✅ Validación completada: {TARGET_FINAL_PER_CATEGORY} registros por categoría, {len(df_final_unified)} en total.")

# Exportación
output_file = f'{CARPETA_PROCESADOS}/dataset_FINAL_UNIFICADO_techmind.csv'
df_final_unified.to_csv(output_file, index=False)

print(f"\n✅ Pipeline de unificación completado. Archivo guardado en: {output_file}")

# Muestra de Auditoría
print("\n=== MUESTRA DE AUDITORÍA (20 registros aleatorios) ===")
display(df_final_unified.sample(min(20, len(df_final_unified)), random_state=42))

=== DISTRIBUCIÓN FINAL DEL DATASET UNIFICADO ===
categoria
Backend           200
Bases de Datos    200
Cloud             200
Data Science      200
Frontend          200
Mobile            200
DevOps            200
Name: count, dtype: int64

✅ Validación completada: 200 registros por categoría, 1400 en total.

✅ Pipeline de unificación completado. Archivo guardado en: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\procesados/dataset_FINAL_UNIFICADO_techmind.csv

=== MUESTRA DE AUDITORÍA (20 registros aleatorios) ===


,titulo,texto,categoria,autor,tipo
665,La etiqueta nutricional del conjunto de datos:...,Los sistemas de inteligencia artificial (IA) c...,Data Science,Sarah Holland,articulo
624,Protección de la inteligencia artificial en el...,Obtenga información sobre cómo las PCs Copilot...,Data Science,Microsoft,módulo
115,Desarrollo de un sistema web inteligente integ...,"A medida que continúa la pandemia de COVID-19,...",Backend,Abid Hassan,articulo
478,Análisis de rendimiento de servicios de comput...,La computación en la nube es un paradigma de i...,Cloud,Alexandru Iosup,articulo
233,Implementación de Microsoft Defender para Storage,Habilite y configure Microsoft Defender para S...,Bases de Datos,Microsoft,módulo
724,SQL para ciencia de datos,A medida que la recopilación de datos ha aumen...,Data Science,"University of California, Davis",articulo
208,¿Es mejor utilizar varias bases de datos con u...,Después de este comentario a una de mis pregun...,Bases de Datos,Strae,apunte
867,"Múltiples entornos (Staging, QA, producción, e...",¿Qué se considera una buena práctica con K8S p...,Frontend,Yoanis Gil,apunte
939,Implementación de microfrontends utilizando co...,Los microfrontends son un estilo arquitectónic...,Frontend,Yuma Nishizu,articulo
479,Un estudio sobre la computación en la nube móv...,RESUMEN Junto con un crecimiento explosivo de ...,Cloud,Hoang T. Dinh,articulo
